# IS 4487 Lab 7

## Outline

Begin where you left on in Lab 6 with the *SuperStore Retail Orders* dataset.
This lab will ask you to do a deeper data exploration

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Labs/Scripts/lab_07_retailer_eda2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Superstore Data Dictionary

 ORDER VARIABLES:
 - Order ID
 - Order Date
 - Order Year Month - Year and month of the order
 - Order Type - Was the order completed at a store or online? (Retail, Online)
 - Quantity - Quantity ordered for the product

 CUSTOMER VARIABLES:
 - Customer Name
 - City
 - State Province
 - Email

PRODUCT VARIABLES:
 - Product Name
 - Product Line - Category of the product (i.e. Bikes Phones)
 - Product Price - Price in US Dollars
 - Product Status - Current status of the product (Active, Inactive)

## Load Libraries

➡️ Assignment Tasks
- Load any necessary libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Data into Dataframe

➡️ Assignment Tasks
- Import data from the SuperStore retail dataset into a dataframe (in GitHub go to Labs > DataSets)
- Describe or profile the dataframe

In [10]:
df = pd.read_csv('https://github.com/Stan-Pugsley/is_4487_base/blob/main/Labs/DataSets/superstore_retail_orders.csv?raw=true')

## Prepare Data

➡️ Assignment Tasks
- Convert any numbers into the correct datatype if they are not already numeric
- Convert any character variables in to categories if they are appropriate for that datatype
- Check for outliers.   Remove any outliers that appear to be mistakes
- Remove rows with empty (NULL) values
- Identify at least one variable with a missing value that can be imputed.   Fill in those empty values.
- Add a "total_amount" variable based on the quantity and price

In [20]:
#data preparation
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56043 entries, 0 to 56042
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          56043 non-null  int64         
 1   order_date        55956 non-null  datetime64[ns]
 2   order_year_month  56011 non-null  period[M]     
 3   customer_name     56043 non-null  object        
 4   city              55569 non-null  object        
 5   state_province    55569 non-null  category      
 6   email             56043 non-null  object        
 7   order_type        56043 non-null  category      
 8   quantity          56043 non-null  int64         
 9   product_name      56043 non-null  object        
 10  product_line      56043 non-null  category      
 11  product_price     56043 non-null  float64       
 12  product_status    56043 non-null  category      
dtypes: category(4), datetime64[ns](1), float64(1), int64(2), object(4), period[M

In [21]:
#converting order_date and order_year_month from string to datetime datatypes
df['order_date'] = pd.to_datetime(df['order_date'], format='%Y-%m-%d', errors='coerce')
df['order_year_month'] = pd.PeriodIndex(df['order_year_month'], freq='M')
#converting some objects into categories
df['product_line'] = df['product_line'].astype('category')
df['product_status'] = df['product_status'].astype('category')
df['order_type'] = df['order_type'].astype('category')
df['state_province'] = df['state_province'].astype('category')
#converting quantity to integer
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').fillna(0).astype('int64')

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56043 entries, 0 to 56042
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          56043 non-null  int64         
 1   order_date        55956 non-null  datetime64[ns]
 2   order_year_month  56011 non-null  period[M]     
 3   customer_name     56043 non-null  object        
 4   city              55569 non-null  object        
 5   state_province    55569 non-null  category      
 6   email             56043 non-null  object        
 7   order_type        56043 non-null  category      
 8   quantity          56043 non-null  int64         
 9   product_name      56043 non-null  object        
 10  product_line      56043 non-null  category      
 11  product_price     56043 non-null  float64       
 12  product_status    56043 non-null  category      
dtypes: category(4), datetime64[ns](1), float64(1), int64(2), object(4), period[M

In [27]:
#using previous lab code to clean up outliers
df_clean = df.copy()

df_clean = df_clean[
    (df_clean['quantity'] <= 80) &
    (df_clean['product_price'] <= 6000) &
    (df_clean['order_date'].dt.year >= 2000) &
    (df_clean['order_date'].dt.year <= 2025)
]
print(df_clean.describe())

           order_id                     order_date      quantity  \
count  54473.000000                          54473  54473.000000   
mean   61604.430984  2023-12-15 18:59:19.620362496      1.489343   
min    45079.000000            2022-01-01 00:00:00      0.000000   
25%    55608.000000            2023-10-05 00:00:00      1.000000   
50%    61821.000000            2024-01-10 00:00:00      1.000000   
75%    68019.000000            2024-04-10 00:00:00      2.000000   
max    74146.000000            2024-06-30 00:00:00     20.000000   
std     7502.794971                            NaN      0.616273   

       product_price  
count   54473.000000  
mean      616.515416  
min         2.290000  
25%         7.950000  
50%       475.600000  
75%       914.620000  
max      3578.270000  
std       817.776413  


In [28]:
#checking for null data
print(df_clean.isnull().sum())
#imputing state_province data
#adding 'Unknown' to the category of 'state_province' and imputing null with 'Unknown'
df_clean['state_province'] = df_clean['state_province'].cat.add_categories(['Unknown'])
df_clean['state_province'] = df_clean['state_province'].fillna('Unknown')
#imputing null city data with 'Unknown'
df_clean['city'] = df_clean['city'].fillna('Unknown')
#verufying imputation
print(df_clean.isnull().sum())

order_id              0
order_date            0
order_year_month      0
customer_name         0
city                469
state_province      469
email                 0
order_type            0
quantity              0
product_name          0
product_line          0
product_price         0
product_status        0
dtype: int64
order_id            0
order_date          0
order_year_month    0
customer_name       0
city                0
state_province      0
email               0
order_type          0
quantity            0
product_name        0
product_line        0
product_price       0
product_status      0
dtype: int64


After imputation, no null values were found so I did not use .dropna() because it would be redundant.

In [29]:
#total_amount variable creation
df_clean['total_amount'] = df_clean['product_price'] * df_clean['quantity']

## Prepare Data - Continued

➡️ Assignment Tasks
- Create a variable called "complete_customer_info".   Use "1" for True and "2" for False.  All customer fields must be valid and not empty before this variable is True.
- Create a bar chart showing the count of customers with complete informaiton versus incomplete information

In [ ]:
#create variable

In [ ]:
#create chart

## Prepare Data - Continued

➡️ Assignment Tasks
- Create seasonal buckets for Winter, Spring, Summer, Fall
- Create a chart to show revenue by season
- Create a chart to show revenue by year

In [ ]:
#seasonality

In [ ]:
#revenue over time

## Prepare Data - Continued

➡️ Assignment Tasks
- Create a variable that will group product lines into "Outdoor" versus "Indoor" products.
- Create a plot to show the correlation between outdoor/indoor versus season  

In [ ]:
#indoor/outdoor variable